In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from jetnet.datasets import JetNet

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

plt.rcParams["text.usetex"] = True
sns.set_palette("deep")

In [ ]:
import pickle
with open("data/x_train.pkl", "rb") as f:
    X_train = pickle.load(f)
with open("data/x_test.pkl", "rb") as f:
    X_test = pickle.load(f)

# Only take global jet features
X_train = X_train[:][1]
X_test = X_test[:][1]

In [ ]:
jet_type_map = {
    0: "Gluons",
    1: "Light quarks",
    2: "Top quarks",
    3: "W bosons",
    4: "Z bosons"
}
sns.histplot(X_train[:, -1].numpy())
plt.xticks(ticks=[0, 1, 2, 3, 4], labels=jet_type_map.values())
# plt.xlabel("Jet Type")
plt.ylabel("Count")
plt.title("Jet type distribution in training set")

In [ ]:
# equalize the number of jets per type
X_train_equalized = []
min_jets_per_type = min(len(X_train[X_train[:, -1] == jet_type]) for jet_type in jet_type_map.keys())
for jet_type in jet_type_map.keys():
    X_train_equalized.extend(X_train[X_train[:, -1] == jet_type][:min_jets_per_type])

X_train_original = X_train.clone()
X_train = torch.tensor(np.array(X_train_equalized), dtype=torch.float32)
print(X_train_original.shape, X_train.shape)

In [ ]:
import os
os.makedirs("gen", exist_ok=True)
os.makedirs("gen/figs", exist_ok=True)
os.makedirs("gen/logs", exist_ok=True)

In [ ]:
sns.histplot(X_train[:, -1].numpy(), alpha=0.5, label="Equalized")
sns.histplot(X_train_original[:, -1].numpy(), alpha=0.5, label="Original")
plt.axhline(min_jets_per_type, linestyle='--', label=f"Equalized jet count ({min_jets_per_type}) ")
plt.xticks(ticks=[0, 1, 2, 3, 4], labels=jet_type_map.values())
plt.legend(loc="lower right", fontsize=12)
plt.ylabel("Count")
plt.savefig("gen/figs/jet_type_distribution_equalized.png", dpi=300, bbox_inches="tight")

In [ ]:
sns.set_palette("deep")
def noise_num_particles(X, noise_std=0.25):
    """
    Add noise to the number of particles in the jets.
    
    Args:
        X (torch.Tensor): Input tensor with jet features.
        noise_std (float): Standard deviation of the noise to be added.
        
    Returns:
        torch.Tensor: Tensor with noisy number of particles.
    """
    num_particles = X[:, 3]
    noise = torch.randn_like(num_particles) * noise_std
    noise = torch.clamp(noise, -1, 1)
    return num_particles + noise
n_bins = 200
std = 0.15
num_particles = X_train[:, 3]
noised_particles = noise_num_particles(X_train, noise_std=std)
sns.histplot(num_particles.numpy(), bins=200, kde=True, alpha=0.5, label="Original particle count", linestyle='--')
sns.histplot(noised_particles.numpy(), bins=200, kde=True, alpha=0.5, label="Noised particle count")
plt.xlabel("Number of particles")
plt.ylabel("Count")
plt.legend()
plt.title(f"Original and noised particle counts with std={std}")
plt.savefig("gen/figs/num_particles_in_jets.png", dpi=300, bbox_inches="tight")

In [ ]:
filtered_noised_particles = noised_particles[(noised_particles > 40) & (noised_particles < 60)]
filtered_original_particles = num_particles[(num_particles > 40) & (num_particles < 60)]
sns.histplot(filtered_original_particles.numpy(), label="Original particle count", bins=n_bins)
sns.histplot(filtered_noised_particles.numpy(), label="Noised particle count", bins=n_bins)
plt.legend()
plt.xlabel("Number of particles")
plt.xticks(ticks=np.arange(40, 61, 2))
plt.title(f"Sample of noised particle count ({n_bins} bins)")
plt.savefig("gen/figs/dequantized_num_particles.png", dpi=300, bbox_inches="tight")

In [ ]:
X_train[:, 3] = noised_particles
X_train.shape

In [ ]:
X_train = X_train.to(device)
X_test = X_test.to(device)

In [ ]:
JET_FEATURES = [r"$\eta$", r"$p_T$", r"$m$", r"$N_P$"]
fig, axs = plt.subplots(1, 4, figsize=(20, 4))
for i, feature in enumerate(JET_FEATURES):
    ax = axs[i]
    for jet_type in jet_type_map.keys():
        sns.kdeplot(
            X_train[X_train[:, -1] == jet_type][:, i].numpy(),
            ax=ax,
            label=jet_type_map[jet_type]
        )
    ax.set_xlabel(feature, fontsize=18)
    if i == 0:
        ax.set_ylabel("Density")
    else:
        ax.set_ylabel("")
legend_handles, legend_labels = axs[0].get_legend_handles_labels()
fig.legend(
    legend_handles,
    legend_labels,
    title="",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=12,
    title_fontsize=13
)
plt.tight_layout()
plt.suptitle("Training set jet attributes", fontsize=18, y=1.05)
plt.savefig("gen/figs/real_jet_attributes.png", dpi=300, bbox_inches="tight")

We will train a normalizing flow conditioned on jet type that generates the jet $\eta$-coordinate, jet transverse momentum $p_T$, jet mass $m$, and the number of particles $N$.

In [ ]:
from util.jet_attributes import one_hot_enc_jet_type, one_hot_to_type

long_types = X_train[:, -1].long()
one_hot_jets = one_hot_enc_jet_type(long_types)
print(one_hot_jets.shape, one_hot_jets.sum(dim=0))

In [ ]:
import normflows as nf
from tqdm.notebook import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
K = 6

latent_size = 4
context_size = 5
hidden_units = 128
hidden_layers = 8

flows = []
for i in range(K):
    flows += [nf.flows.AutoregressiveRationalQuadraticSpline(latent_size, hidden_layers, hidden_units, num_context_channels=context_size)]
    flows += [nf.flows.LULinearPermute(latent_size)]

q0 = nf.distributions.DiagGaussian(latent_size, trainable=False)
model = nf.ConditionalNormalizingFlow(q0, flows).to(device)
model

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
total_params

In [ ]:
torch.manual_seed(RANDOM_SEED)
max_iter = 35_000
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
batch_size = 4096

loss_hist = np.array([])
for it in tqdm(range(max_iter)):
    optimizer.zero_grad()

    indices = torch.randperm(len(X_train), device=device)[:batch_size]
    jets = X_train[indices]
    jet_info = jets[:, :-1]  # Exclude the jet type column
    jet_types = jets[:, -1].long()  # Get the jet type column
    one_hot_types = one_hot_enc_jet_type(jet_types).to(device)
    
    # Compute loss
    loss = model.forward_kld(jet_info, one_hot_types)

    # Do backprop and optimizer step
    if ~(torch.isnan(loss) | torch.isinf(loss)):
        loss.backward()
        optimizer.step()

    # Log loss
    loss_hist = np.append(loss_hist, loss.to('cpu').data.numpy())

# Plot loss
plt.figure(figsize=(10, 10))
plt.plot(loss_hist, label='loss')
plt.legend()
plt.show()

In [ ]:
sns.set_palette("deep")
sns.lineplot(
    x=np.arange(len(loss_hist)),
    y=loss_hist,
    label="Loss",
)
plt.xlabel("Epoch", fontsize=14)
plt.ylabel("Forward KL Divergence Loss", fontsize=14)
plt.savefig("gen/figs/jet_attr_nf_loss.png", dpi=300, bbox_inches="tight")

In [ ]:
model.eval()
n_samples = 50_000
sample_jet_types = torch.randint(0, 5, (n_samples,)).to(device)
one_hot_types = one_hot_enc_jet_type(sample_jet_types)
print(one_hot_types.shape, one_hot_types.sum(dim=0))

In [ ]:
sample_vals, sample_logprobs = model.sample(n_samples, context=one_hot_types)
sample_jet_types.shape, sample_vals.shape, sample_logprobs.shape

# Quantize the number of particles
sample_vals[:, 3] = torch.round(sample_vals[:, 3])
sample_vals[:, 3]

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4))
for i, feature in enumerate(JET_FEATURES):
    ax = axs[i]
    sns.kdeplot(
        sample_vals[:, i].detach().numpy(),
        ax=ax,
        label=r"Generated " + feature + r" (all jets)",
    )
    sns.kdeplot(
        X_test[:, i].numpy(),
        ax=ax,
        label=r"Test " + feature + r" (all jets)",
    )
    ax.set_xlabel(feature, fontsize=18)
    if i == 0:
        ax.set_ylabel("Density")
    else:
        ax.set_ylabel("")
    ax.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.savefig("gen/figs/jet_attr_nf_sampled_all_features.png", dpi=300, bbox_inches="tight")


In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 8))
for i, feature in enumerate(JET_FEATURES):
    ax = axs[0, i]
    for jet_type in jet_type_map.keys():
        sns.kdeplot(
            X_test[X_test[:, -1] == jet_type][:, i].numpy(),
            ax=ax,
            label=jet_type_map[jet_type]
        )
    # ax.set_xlabel(feature, fontsize=18)
    if i == 0:
        ax.set_ylabel("Test set jet attributes", fontsize=18, rotation=0, labelpad=60)
    else:
        ax.set_ylabel("")

    ax = axs[1, i]
    for jet_type in jet_type_map.keys():
        sns.kdeplot(
            sample_vals[sample_jet_types == jet_type][:, i].detach().numpy(),
            ax=ax,
            label=jet_type_map[jet_type]
        )
    ax.set_xlabel(feature, fontsize=18)
    if i == 0:
        ax.set_ylabel("Generated jet attributes", fontsize=18, rotation=0, labelpad=60)
    else:
        ax.set_ylabel("")

for i in range(4):
    ax1 = axs[0, i]
    ax2 = axs[1, i]

    # Get combined x and y limits for both axes
    xlims = np.array([ax1.get_xlim(), ax2.get_xlim()])
    ylims = np.array([ax1.get_ylim(), ax2.get_ylim()])

    # Set both axes to have the same limits
    xlim_combined = (xlims[:, 0].min(), xlims[:, 1].max())
    ylim_combined = (ylims[:, 0].min(), ylims[:, 1].max())

    ax1.set_xlim(xlim_combined)
    ax2.set_xlim(xlim_combined)
    ax1.set_ylim(ylim_combined)
    ax2.set_ylim(ylim_combined)


legend_handles, legend_labels = axs[0, 0].get_legend_handles_labels()
fig.legend(
    legend_handles,
    legend_labels,
    title="",
    loc="center right",
    bbox_to_anchor=(1.05, 0.5),
    fontsize=14,
    title_fontsize=13
)
plt.tight_layout()
plt.savefig("gen/figs/jet_attr_nf_sampled_jet_type.png", dpi=300, bbox_inches="tight")

In [ ]:
from jetnet.evaluation import fpd, kpd
fpd_val, fpd_err = fpd(
    real_features=X_test[:, :-1].numpy(),
    gen_features=sample_vals.detach().numpy(),
    seed=RANDOM_SEED
)
kpd_val, kpd_err = kpd(
    real_features=X_test[:, :-1].numpy(),
    gen_features=sample_vals.detach().numpy(),
    seed=RANDOM_SEED
)
print(f"FPD: {fpd_val:.5E} ± {fpd_err:.5E}")
print(f"KPD: {kpd_val:.5E} ± {kpd_err:.5E}")
with open("gen/logs/jet_attr_nf_fpd.txt", "w") as f:
    f.write(f"FPD: {fpd_val:.5E} ± {fpd_err:.5E}\n")
    f.write(f"Seed: {RANDOM_SEED}\n")
    f.write(f"Number of samples: {n_samples}\n")
    f.write(f"Batch size: {batch_size}\n")
    f.write(f"Hidden units: {hidden_units}\n")
    f.write(f"Hidden layers: {hidden_layers}\n")
    f.write(f"Latent size: {latent_size}\n")
    f.write(f"Number of flows: {K}\n")
    f.write(f"Max iterations: {max_iter}\n")
    f.close()

In [ ]:
with open("upload/jet_attr_nf_model.pkl", "wb") as f:
    pickle.dump(model, f)